In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
import sys
sys.path.append("01-agentic-rag/")
sys.path.append("04-evaluation/")
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [28]:
pd.DataFrame(documents)#[lambda x: x.id == '4487db3924']

,id,course,section,question,answer
0,74eb249bbf,llm-zoomcamp,General Course-Related Questions,I just discovered the course. Can I still join?,"Yes, but if you want to receive a certificate,..."
1,977bf7786c,llm-zoomcamp,General Course-Related Questions,Course: I have registered for the LLM Zoomcamp...,You don't need it. You're accepted. You can al...
2,489dd1c9d9,llm-zoomcamp,General Course-Related Questions,What is the video/zoom link to the stream for ...,The zoom link is only published to instructors...
3,04919992b3,llm-zoomcamp,General Course-Related Questions,How should I start the course and follow the w...,Start with the [LLM Zoomcamp docs](https://dat...
4,c2903069a0,llm-zoomcamp,General Course-Related Questions,Leaderboard: I am not on the leaderboard / how...,"When you set up your account, you are automati..."
...,...,...,...,...,...
148,dd53e420ef,llm-zoomcamp,Workshop: Open-Source Data Ingestion (dlt),What needs to change in `agent.py` to use Gemi...,"In the dlt workshop's `homework/agent.py`, cha..."
149,8ede00dc4c,llm-zoomcamp,Workshop: Open-Source Data Ingestion (dlt),Why does the dlt workshop pipeline load fewer ...,Logfire may not have exported and indexed the ...
150,8ca99429d0,llm-zoomcamp,Workshop: Open-Source Data Ingestion (dlt),"How do I fix DuckDB's ""Ambiguous reference to ...","When you use `destination=""duckdb""`, dlt norma..."
151,e00ceb24be,llm-zoomcamp,Workshop: Open-Source Data Ingestion (dlt),Where are the current dlt workshop materials a...,The current workshop materials are in the\n[`c...


In [18]:
ground_truth

[{'question': 'Where exactly do I find the course files and homework for my week in the GitHub repo?',
  'document': '4487db3924'},
 {'question': 'Is there a folder that already puts all the week 1, week 2, week 3, and week 4 stuff together for the 2026 cohort?',
  'document': '4487db3924'},
 {'question': 'Where are the lectures and updated workshop videos posted, and do I need to wait for announcements before checking them?',
  'document': '4487db3924'},
 {'question': 'What happens if I miss a homework deadline — does the submission form stay open or close after the due date?',
  'document': '4487db3924'},
 {'question': 'How can I get extra points in this course, and what do I need to tag when I share my work online?',
  'document': '4487db3924'},
 {'question': 'Are the classes actually live, or are they mostly recorded videos I can watch later?',
  'document': 'd8385e12f6'},
 {'question': 'If the lectures are pre-recorded, where can I find the videos for this course?',
  'document': 

In [4]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [10]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [17]:
q = ground_truth[3]
print(q["question"])
compute_relevance_text(q)

What happens if I miss a homework deadline — does the submission form stay open or close after the due date?


[0, 0, 0, 0, 0]